In [ ]:
import cupy as cp
import matplotlib.pyplot as plt
import pandas as pd
import time
import cv2
import xraylib
import random
import cupyx.scipy.ndimage as ndimage
from utils import *


%matplotlib inline

## Sizes and propagation settings

In [ ]:
# sizes
n = 1024  # detector size in each dimension
pad = 32 # padding for reconstruction
npsi = n+2*pad # object size
ncode = n*3//2 # size of the CA, should be bigger than the object since we move CA
npos = 8 # number of position for the CA


# propagation parameters
detector_pixelsize = 3.03751e-6 # [m] detector pixel size
energy = 33.35  # [keV] xray energy
wavelength = 1.24e-09/energy  # [m] wave length
focusToDetectorDistance = 1.28  # [m]
z1 = 4.3e-3 # [m] position of the sample
z1c = -17.5e-3 # [m] position of the CA
z2 = focusToDetectorDistance-z1
distance = (z1*z2)/focusToDetectorDistance
distancec = (z1-z1c)/(z1c/z1)
magnification = focusToDetectorDistance/z1
magnifications2 = z1/z1c
voxelsize = np.abs(detector_pixelsize/magnification)  # object voxel size
ex = 16 # extra pad for extracting patches

## Generate positions of the CA

In [ ]:
cp.random.seed(2)
pos = (cp.random.random([npos,2]).astype('float32')-0.5)*n/4
mplot_positions(pos)

## Load probe

In [ ]:
!wget -nc https://g-110014.fd635.8443.data.globus.org/holotomocupy/examples_synthetic/data/prb_id16a/prb_abs_2048.tiff 
!wget -nc https://g-110014.fd635.8443.data.globus.org/holotomocupy/examples_synthetic/data/prb_id16a/prb_phase_2048.tiff 

q_abs = read_tiff(f'prb_abs_2048.tiff')[0,1024-npsi//2:1024+npsi//2,1024-npsi//2:1024+npsi//2]
q_phase = read_tiff(f'prb_phase_2048.tiff')[0,1024-npsi//2:1024+npsi//2,1024-npsi//2:1024+npsi//2]
q = cp.array(q_abs*np.exp(1j*q_phase)).astype('complex64')

#smooth it slightly
v = cp.arange(-npsi//2,npsi//2)/npsi
[vx,vy] = cp.meshgrid(v,v)
v = cp.exp(-10*(vx**2+vy**2))
fq = cp.fft.fftshift(cp.fft.fftn(cp.fft.fftshift(q)))
q = cp.fft.fftshift(cp.fft.ifftn(cp.fft.fftshift(fq*v))).astype('complex64')
mshow_polar(q)


## Generate an object (Siemens star)

In [ ]:
img = np.zeros((npsi, npsi, 3), np.uint8)
triangle = np.array([(1.5*npsi//16, npsi//2-npsi//32), (1.5*npsi//16, npsi//2+npsi//32), (npsi//2-npsi//128, npsi//2)], np.float32)
star = img[:,:,0]*0
for i in range(0, 360, 15):
    img = np.zeros((npsi, npsi, 3), np.uint8)
    degree = i
    theta = degree * np.pi / 180
    rot_mat = np.array([[np.cos(theta), -np.sin(theta)],
                        [np.sin(theta), np.cos(theta)]], np.float32)    
    rotated = cv2.gemm(triangle-npsi//2, rot_mat, 1, None, 1, flags=cv2.GEMM_2_T)+npsi//2
    cv2.fillPoly(img, [np.int32(rotated)], (255, 0, 0))
    star+=img[:,:,0]
[x,y] = np.meshgrid(np.arange(-npsi//2,npsi//2),np.arange(-npsi//2,npsi//2))
x = x/npsi*2
y = y/npsi*2
# add holes in triangles
circ = (x**2+y**2>0.355)+(x**2+y**2<0.345)
circ *= (x**2+y**2>0.083)+(x**2+y**2<0.08)
circ *= (x**2+y**2>0.0085)+(x**2+y**2<0.008)
star = star*circ/255

v = np.arange(-npsi//2,npsi//2)/npsi
[vx,vy] = np.meshgrid(v,v)
v = np.exp(-5*(vx**2+vy**2))
fu = np.fft.fftshift(np.fft.fftn(np.fft.fftshift(star)))
star = np.fft.fftshift(np.fft.ifftn(np.fft.fftshift(fu*v))).real


delta = 1-xraylib.Refractive_Index_Re('Au',energy,19.3)
beta = xraylib.Refractive_Index_Im('Au',energy,19.3)

thickness = 1e-6/voxelsize # siemens star thickness in pixels
# form Transmittance function
u = star*(-delta+1j*beta) # note -delta
Ru = u*thickness 
psi = np.exp(1j * Ru * voxelsize * 2 * np.pi / wavelength)[np.newaxis].astype('complex64')
psi = cp.array(psi[0])


mshow_polar(psi,True)

## generate coded aperture

In [ ]:
bin_size = 2e-6
code_thickness = 1.5e-6

random.seed(10)
nill = 2*ncode
ill_global = cp.zeros([nill,nill],dtype='bool')
ill0 = cp.zeros([nill*nill],dtype='bool')
ill_ids = random.sample(range(0, nill*nill), int(nill*nill*0.55))
ill0[ill_ids] = 1
ill_global = ill0.reshape(nill,nill)

# form codes for simulations
nill = int(ncode*voxelsize/np.abs(magnifications2)//(bin_size*2))*2
ill = cp.zeros([nill,nill],dtype='bool')
ill0 = ill_global
ill = ill0[ill0.shape[0]//2-nill//2:ill0.shape[0]//2+(nill)//2,
                ill0.shape[1]//2-nill//2:ill0.shape[1]//2+(nill)//2]#.reshape(nill,nill)

ill = ndimage.zoom(ill,ncode/nill,order=0,grid_mode=True,mode='grid-wrap')

delta = 1-xraylib.Refractive_Index_Re('Au',energy,19.3)
beta = xraylib.Refractive_Index_Im('Au',energy,19.3)

thickness = code_thickness/voxelsize # thickness in pixels

# form Transmittance function
Rill = ill*(-delta+1j*beta)*thickness 
Rill=ndimage.rotate(Rill, 45, axes=(1, 0), reshape=False, order=3, mode='reflect',
                   prefilter=True)

v = cp.arange(-ncode//2,ncode//2)/2/ncode
[vx,vy] = cp.meshgrid(v,v)
v = cp.exp(-10*(vx**2+vy**2))
fill = cp.fft.fftshift(cp.fft.fftn(cp.fft.fftshift(Rill)))
Rill = cp.fft.fftshift(cp.fft.ifftn(cp.fft.fftshift(fill*v)))
Rill = Rill.astype('complex64')

code = cp.exp(1j * Rill * voxelsize * 2 * np.pi / wavelength).astype('complex64')

mshow_polar(code,True)
mshow_polar(code[:n//5,:n//5],True)



## Fresnel kernels for porpagating between the sample and detector, and between the CA and sample

In [ ]:

fx = cp.fft.fftfreq(npsi * 2, d=voxelsize).astype("float32")
[fx, fy] = cp.meshgrid(fx, fx)
fker = cp.exp(-1j * cp.pi * wavelength * distance * (fx**2 + fy**2))
mshow_complex(cp.fft.fftshift(fker),True)

fx = cp.fft.fftfreq(npsi * 2, d=voxelsize).astype("float32")
[fx, fy] = cp.meshgrid(fx, fx)
fkerc = cp.exp(-1j * cp.pi * wavelength * distancec * (fx**2 + fy**2))
mshow_complex(cp.fft.fftshift(fkerc),True)

## Construct operators

In [ ]:
def E(psi, ri):
    """Extract patches"""

    stx = ncode // 2 - ri[:, 1] - (npsi+2*ex) // 2
    sty = ncode // 2 - ri[:, 0] - (npsi+2*ex) // 2
    res = cp.empty([len(stx), (npsi+2*ex), (npsi+2*ex)], dtype="complex64")
    Efast_kernel(
        (
            int(cp.ceil((npsi+2*ex) / 16)),
            int(cp.ceil((npsi+2*ex) / 16)),
            int(cp.ceil(len(stx) / 4)),
        ),
        (16, 16, 4),
        (res, psi, stx, sty, len(stx), (npsi+2*ex), ncode),
    )
    return res


def ET(psi, psir, ri):
    """Adjoint extract patches"""

    stx = ncode // 2 - ri[:, 1] - (npsi+2*ex) // 2
    sty = ncode // 2 - ri[:, 0] - (npsi+2*ex) // 2
    ETfast_kernel(
        (
            int(cp.ceil((npsi+2*ex) / 16)),
            int(cp.ceil((npsi+2*ex) / 16)),
            int(cp.ceil(len(stx) / 4)),
        ),
        (16, 16, 4),
        (psir, psi, stx, sty, len(stx), (npsi+2*ex), ncode),
    )
    return psi


def S(psi, ri, r):
    """Extract patches with subpixel shift"""
    psir = E(psi, ri)

    x = cp.fft.fftfreq((npsi+2*ex)).astype("float32")
    [y, x] = cp.meshgrid(x, x)
    tmp = cp.exp(
        -2 * cp.pi * 1j * (y * r[:, 1, None, None] + x * r[:, 0, None, None])
    ).astype("complex64")
    psir = cp.fft.ifft2(tmp * cp.fft.fft2(psir))
    psir = psir[:, ex : (npsi+2*ex) - ex, ex : (npsi+2*ex) - ex]
    return psir


def ST(d, ri, r):
    """Adjont extract patches with subpixel shift"""
    psi = cp.zeros([ncode, ncode], dtype="complex64")
    psir = cp.pad(d, ((0, 0), (ex, ex), (ex, ex)))

    x = cp.fft.fftfreq((npsi+2*ex)).astype("float32")
    [y, x] = cp.meshgrid(x, x)
    tmp = cp.exp(
        2 * cp.pi * 1j * (y * r[:, 1, None, None] + x * r[:, 0, None, None])
    ).astype("complex64")
    psir = cp.fft.ifft2(tmp * cp.fft.fft2(psir))

    ET(psi, psir, ri)
    return psi

def Dc(psi):
    """Forward propagator: CA-> sample"""

    ff = psi.copy()
    ff = fwd_pad(ff)
    v = cp.ones(2*npsi,dtype='float32')
    v[:npsi//2] = cp.sin(cp.linspace(0,1,npsi//2)*cp.pi/2)
    v[-npsi//2:] = cp.cos(cp.linspace(0,1,npsi//2)*cp.pi/2)
    v = cp.outer(v,v)
    ff *= v*2        
    ff = cp.fft.ifft2(cp.fft.fft2(ff)*fkerc)
    ff = ff[:,npsi//2:-npsi//2,npsi//2:-npsi//2]
    return ff


def DcT(psi):
    """Adjoint propagator: sample-> CA"""

    ff = cp.pad(psi,((0,0),(npsi//2,npsi//2),(npsi//2,npsi//2)))
    ff = cp.fft.ifft2(cp.fft.fft2(ff)/fkerc)
    v = cp.ones(2*npsi,dtype='float32')
    v[:npsi//2] = cp.sin(cp.linspace(0,1,npsi//2)*cp.pi/2)
    v[-npsi//2:] = cp.cos(cp.linspace(0,1,npsi//2)*cp.pi/2)
    v = cp.outer(v,v)
    ff *= v*2        
    ff = adj_pad(ff)
    return ff

def D(psi):
    """Forward propagator: sample->detector"""

    ff = psi.copy()
    ff = fwd_pad(ff)
    v = cp.ones(2*npsi,dtype='float32')
    v[:npsi//2] = cp.sin(cp.linspace(0,1,npsi//2)*cp.pi/2)
    v[-npsi//2:] = cp.cos(cp.linspace(0,1,npsi//2)*cp.pi/2)
    v = cp.outer(v,v)
    ff *= v*2        
    ff = cp.fft.ifft2(cp.fft.fft2(ff)*fker)
    ff = ff[:,npsi//2:-npsi//2,npsi//2:-npsi//2]
    # crop to detector size
    ff = ff[:, pad : npsi - pad, pad : npsi - pad]
    return ff

def DT(psi):
    """Adjoint propagator: detector->sample"""

    # pad to the probe size
    ff = cp.pad(psi, ((0, 0), (pad, pad), (pad, pad)))
    ff = cp.pad(ff,((0,0),(npsi//2,npsi//2),(npsi//2,npsi//2)))
    ff = cp.fft.ifft2(cp.fft.fft2(ff)/fker)
    v = cp.ones(2*npsi,dtype='float32')
    v[:npsi//2] = cp.sin(cp.linspace(0,1,npsi//2)*cp.pi/2)
    v[-npsi//2:] = cp.cos(cp.linspace(0,1,npsi//2)*cp.pi/2)
    v = cp.outer(v,v)
    ff *= v*2        
    ff = adj_pad(ff)
    return ff


# adjoint tests
pos_test = 10 * (cp.random.random([npos, 2]) - 0.5).astype("float32")
ri_test = pos_test.astype("int32")
r_test = pos_test - ri_test

arr1 = (cp.random.random([ncode, ncode]) + 1j * cp.random.random([ncode, ncode])).astype(
    "complex64"
)
arr2 = E(arr1, ri_test)
arr3 = arr1 * 0
ET(arr3, arr2, ri_test)
print("<E(psi),E(psi)>==<psi,ET(E(psi)):")
print(f"{cp.sum(arr1 * cp.conj(arr3))}==\n{cp.sum(arr2 * cp.conj(arr2))}\n")

arr1 = (cp.random.random([ncode, ncode]) + 1j * cp.random.random([ncode, ncode])).astype(
    "complex64"
)
arr2 = S(arr1, ri_test, r_test)
arr3 = ST(arr2, ri_test, r_test)
print("<S(psi),S(psi)>==<psi,ST(S(psi)):")
print(f"{cp.sum(arr1 * cp.conj(arr3))}==\n{cp.sum(arr2 * cp.conj(arr2))}\n")

arr1 = (
    cp.random.random([npos, npsi, npsi]) + 1j * cp.random.random([npos, npsi, npsi])
).astype("complex64")
arr2 = Dc(arr1)
arr3 = DcT(arr2)
print("<Dc(psi),Dc(psi)>==<psi,DcT(Dc(psi)):")
print(f"{cp.sum(arr1 * cp.conj(arr3))}==\n{cp.sum(arr2 * cp.conj(arr2))}\n")


arr1 = (
    cp.random.random([npos, npsi, npsi]) + 1j * cp.random.random([npos, npsi, npsi])
).astype("complex64")
arr2 = D(arr1)
arr3 = DT(arr2)
print("<D(psi),D(psi)>==<psi,DT(D(psi)):")
print(f"{cp.sum(arr1 * cp.conj(arr3))}==\n{cp.sum(arr2 * cp.conj(arr2))}\n")

# Model data on the detector
### $d = |D(\psi\cdot D_c(q\cdot S_{r_i+r}(c))) |^2$, where $r_i$ is the close integer part of the position and $r$ is the floating part

In [ ]:
ri = cp.round(pos).astype("int32")
r = pos - ri
data = cp.abs(D(psi*Dc(q*S(code, ri, r)))) ** 2

mshow(data[0],True)


# model data without object

In [ ]:
ri = cp.round(pos).astype("int32")
r = pos - ri
data_ref = cp.abs(D(Dc(q*S(code, ri, r)))) ** 2

mshow(data_ref[0],True)


# data without code and sample

In [ ]:
ri = cp.round(pos).astype("int32")
r = pos - ri
ref = cp.abs(D(Dc(q*S(code*0+1, ri, r)))) ** 2
ref=ref[0]
mshow(ref,True)


In [ ]:
cdata = data/(data_ref+1e-3)
rdata = data/(ref+1e-3)
mshow(cdata[0],True,vmax=5)
mshow(rdata[0],True,vmax=5)

### Initial guess for the object calculated with the Paganin method

In [ ]:
def Paganin(data, wavelength, voxelsize, delta_beta, alpha):
    fx = cp.fft.fftfreq(data.shape[-1], d=voxelsize).astype("float32")
    [fx, fy] = cp.meshgrid(fx, fx)
    rad_freq = cp.fft.fft2(data)
    taylorExp = 1 + wavelength * distance * cp.pi * (delta_beta) * (fx**2 + fy**2)
    numerator = taylorExp * (rad_freq)
    denominator = taylorExp**2 + alpha
    phase = cp.log(cp.real(cp.fft.ifft2(numerator / denominator)))
    phase = delta_beta * 0.5 * phase
    return phase


def rec_init(rdata):
    recMultiPaganin = cp.zeros([npsi, npsi], dtype="float32")
    for j in range(npos):
        
        rpsi = Paganin(rdata[j], wavelength, voxelsize, 10.05, 1e-3)
        rpsi = cp.pad(rpsi,((pad,pad),(pad,pad)))
        recMultiPaganin += rpsi
        
    recMultiPaganin /= npos
    recMultiPaganin = cp.exp(1j * recMultiPaganin)
    return recMultiPaganin


psi_init = rec_init(cdata)
mshow_polar(psi_init,True)

#### Initial guess for the probe calculated by backpropagating the square root of the reference image
#### Smooth the probe borders for stability

In [ ]:
q_init = DT(cp.sqrt(ref[cp.newaxis]))[0]

ppad = 3 * pad // 2
q_init = cp.pad(
    q_init[ppad : npsi - ppad, ppad : npsi - ppad],
    ((ppad, ppad), (ppad, ppad)),
    "symmetric",
)
mshow_polar(q_init,True)

## fixed shifted code ($S_{r_i+r}(c)$)

In [ ]:
scode = S(code,ri,r)

# Reconstruction by the bilinear Hessian (BH) method

### $F(\Psi)=\||\Psi|-d\|_2^2$, where $\Psi = D(\psi\cdot D_c(q\cdot S_{r_i+r}(c)))$ for $\psi$ and $q$ as unknown

### Reused variables to accelerate computations

In [ ]:
def calc_reused(reused, vars):
    """Calculate variables reused during calculations"""
    (q, psi) = (vars["q"], vars["psi"])
    reused["big_psi"] = D(psi*Dc(q*scode))

### Gradient for the Gaussian noise model

##### $$\nabla F|_{\Psi_0}=2D^*\left( \Psi_0-d\cdot\Psi_0/|\Psi_0|\right)$$

### store it with reused variables

In [ ]:
def gradientF(reused, d):
    big_psi = reused["big_psi"]
    td = d * (big_psi / (cp.abs(big_psi)))
    reused["gradF"] = 2*DT(big_psi - td)

### Gradients for the object and probe

##### $$\nabla_{\psi} f|_{(q_0,\psi_0)}= \left(\overline{D_c(J(q_0)\cdot S_{r_i+r_0}(c))}\cdot \nabla F\right)$$

##### $$\nabla_{q} f|_{(q_0,\psi_0)}=J^*\left( \overline{S_{r_i+r_0}(c)}\cdot D_c^*(\overline{\psi}\cdot\nabla F)\right).$$




In [ ]:
def gradient_psi(q, gradF):
    return cp.sum(cp.conj(Dc(q*scode)) * gradF,axis=0)

def gradient_q(psi, gradF):
    return cp.sum(cp.conj(scode) * DcT(cp.conj(psi)*gradF), axis=0)

def gradients(vars, pars, reused):
    (q, psi) = (vars["q"], vars["psi"])
    (gradF) = (reused["gradF"])
    
    dpsi = gradient_psi(q,gradF)
    dq = gradient_q(psi,gradF)
    grads = {"psi": dpsi, "q": dq}
    return grads

## Bilinear hessian for the phase retrieval formulation
##### $$\frac{1}{2}\mathcal{H}|_{\Psi_0}(\Delta\Psi^{(1)},\Delta\Psi^{(2)})= \left\langle \mathbf{1}-d_{0}, \mathsf{Re}({\Delta\Psi^{(1)}}\overline{\Delta\Psi^{(2)}})\right\rangle+\left\langle d_{0},(\mathsf{Re} (\overline{l_0}\cdot \Delta\Psi^{(1)}))\cdot (\mathsf{Re} (\overline{l_0}\cdot \Delta\Psi^{(2)})) \right\rangle$$
##### $$l_0=\Psi_0/|\Psi_0|$$
##### $$d_0=d/|\Psi_0|$$

In [ ]:
def hessian_F(big_psi, dbig_psi1, dbig_psi2, d):
    l0 = big_psi / (cp.abs(big_psi))
    d0 = d / (cp.abs(big_psi))
    v1 = cp.sum((1 - d0) * reprod(dbig_psi1, dbig_psi2))
    v2 = cp.sum(d0 * reprod(l0, dbig_psi1) * reprod(l0, dbig_psi2))
    return 2 * (v1 + v2)

Using theorem (Carlsson):
\begin{equation}
G(x) = F(M(x))  
\end{equation}

\begin{equation}
\begin{aligned}
&M(x_0+\Delta x)=M(x_0)+DM|_{x_0}(\Delta x)+\frac{1}{2}D^2M|_{x_0}(\Delta x,\Delta x)+O(\|\Delta x\|^3)
\end{aligned}
\end{equation}

\begin{equation}\begin{aligned}
H^G|_{x_0}(\Delta x^{(1)},\Delta x^{(2)})=&\Big\langle \nabla F|_{M(x_0)}, D^2M|_{x_0}(\Delta x^{(1)},\Delta x^{(2)})\Big\rangle +\\&H^F|_{M(x_0)}\Big(DM|_{x_0}(\Delta x^{(1)}),DM|_{x_0}(\Delta x^{(2)} )\Big).
\end{aligned}
\end{equation}

In our case:
\begin{equation}
    M(x_0) = M(q,\psi) = \psi\cdot D_c(q\cdot S_{r+r_i}(c))
\end{equation}


In [ ]:
def hessian(q,psi,dq1,dpsi1,dq2,dpsi2,big_psi,gradF,d):        
    dm1 = dpsi1*Dc(q*scode)+psi*Dc(dq1*scode)
    dm2 = dpsi2*Dc(q*scode)+psi*Dc(dq2*scode)    
    d2m1 = dpsi1*Dc(dq2*scode)+dpsi2*Dc(dq1*scode)
    
    Ddm1 = D(dm1)
    Ddm2 = D(dm2)
    return redot(gradF, d2m1) + hessian_F(big_psi, Ddm1, Ddm2, d)

#### Updates with scaling
$$ \eta^{(j)}_z=-\rho_z^2\nabla_z f|_{({x}^{(j)},{{y}}^{(j)})}+\beta_j \eta^{(j)}_{{{z}}}.$$
$$\beta_j = \frac{\mathcal{H}^{ f}|_{({x}^{(j)},{{y}}^{(j)})}\Big((\nabla_{{{{x}}}} \rho_x^2 f|_{({x}^{(j)},{{y}}^{(j)})},\rho_y^2\nabla_{{{{y}}}} f|_{({x}^{(j)},{{y}}^{(j)})}), ( \eta^{(j-1)}_{{{x}}},\eta^{(j-1)}_{{{y}}})\Big)}{\mathcal{H}^{ f}|_{({x}^{(j)},{{y}}^{(j)})}\Big((\eta^{(j-1)}_{{{x}}}, \eta^{(j-1)}_{{{y}}}),(\eta^{(j-1)}_{{{x}}}, \eta^{(j-1)}_{{{y}}})\Big)}$$

\begin{equation}
    \alpha^{(j)}=\frac{\Big\langle(\nabla_{{{{x}}}} f|_{({x}^{(j)},{{y}}^{(j)})},\nabla_{{{{y}}}} f|_{({x}^{(j)},{{y}}^{(j)})}), ( \eta^{(j)}_{{{x}}}, \eta^{(j)}_{{{y}}})\Big\rangle}{\mathcal{H}^{ f}|_{({x}^{(j)},{{y}}^{(j)})}\Big(( \eta^{(j)}_{{{x}}}, \eta^{(j)}_{{{y}}}),( \eta^{(j)}_{{{x}}}, \eta^{(j)}_{{{y}}})\Big)}.
\end{equation} 

In [ ]:
def calc_beta(vars, grads, etas, pars, reused, d):
    (q, psi) = (vars["q"], vars["psi"])
    (big_psi, gradF) = (reused["big_psi"], reused["gradF"])
    rho = pars["rho"]

    # note scaling with rho
    (dpsi1, dq1) = (grads["psi"] * rho[0] ** 2, grads["q"] * rho[1] ** 2)
    (dpsi2, dq2) = (etas["psi"], etas["q"])

    top = hessian(q,psi,dq1,dpsi1,dq2,dpsi2,big_psi,gradF,d)
    bottom = hessian(q,psi,dq2,dpsi2,dq2,dpsi2,big_psi,gradF,d)    
    
    return top / bottom

def calc_alpha(vars, grads, etas, pars, reused, d):
    (q, psi) = (vars["q"], vars["psi"])
    (big_psi, gradF) = (reused["big_psi"], reused["gradF"])
    (dpsi1, dq1) = (grads["psi"], grads["q"])
    (dpsi2, dq2) = (etas["psi"], etas["q"])
    
    top = -redot(dpsi1, dpsi2) - redot(dq1, dq2)    
    bottom = hessian(q,psi,dq2,dpsi2,dq2,dpsi2,big_psi,gradF,d)    

    return top / bottom, top, bottom

### Debug functions

In [ ]:
def minF(big_psi, d):
    return cp.linalg.norm(cp.abs(big_psi) - d) ** 2

def plot_debug(vars, etas, pars, top, bottom, alpha, data, i):
    """Check the minimization functional behaviour"""
    if i % pars["vis_step"] == 0 and pars["vis_step"] != -1:
        (q, psi) = (vars["q"], vars["psi"])
        (dq2, dpsi2) = (etas["q"], etas["psi"])
        rho = pars["rho"]

        npp = 13
        errt = cp.zeros(npp * 2)
        errt2 = cp.zeros(npp * 2)
        for k in range(0, npp * 2):
            psit = psi + (alpha * k / (npp - 1)) * dpsi2
            qt = q + (alpha * k / (npp - 1)) * dq2
            errt[k] = minF(D(psit*Dc(qt*scode)),data)

        t = alpha * (cp.arange(2 * npp)) / (npp - 1)
        errt2 = minF(D(psi*Dc(q*scode)),data)
        errt2 = errt2 - top * t + 0.5 * bottom * t**2

        plt.plot(
            alpha.get() * cp.arange(2 * npp).get() / (npp - 1),
            errt.get(),
            ".",
            label="approximation",
        )
        plt.plot(
            alpha.get() * cp.arange(2 * npp).get() / (npp - 1),
            errt2.get(),
            ".",
            label="real",
        )
        plt.legend()
        plt.grid()
        plt.show()


def vis_debug(vars, pars, i):
    """Visualization and data saving"""
    if i % pars["vis_step"] == 0 and pars["vis_step"] != -1:
        (q, psi) = (vars["q"], vars["psi"])
        mshow_polar(psi,True)
        mshow_polar(q,True)


def error_debug(vars, pars, reused, data, alpha, i):
    """Visualization and data saving"""
    if i % pars["err_step"] == 0 and pars["err_step"] != -1:
        err = minF(reused["big_psi"], data)
        print(f"{i}) {err=:1.5e} {alpha=:1.5e}", flush=True)
        vars["table"].loc[len(vars["table"])] = [i, err.get(), alpha.get(), time.time()]
        # vars["table"].to_csv(f"{pars['flg']}", index=False)

In [ ]:
def BH(data, vars, pars):
    # sqrt data first
    data = cp.sqrt(data)

    rho = pars["rho"]
    reused = {}
    for i in range(pars["niter"]):
        # Calc reused variables and gradF
        calc_reused(reused, vars)
        gradientF(reused, data)

        # debug and visualization        
        vis_debug(vars, pars, i)

        # gradients for each variable
        grads = gradients(vars, pars, reused)

        if i == 0:
            etas = {}
            etas["psi"] = -grads["psi"] * rho[0] ** 2
            etas["q"] = -grads["q"] * rho[1] ** 2
        else:
            # conjugate direction
            beta = calc_beta(vars, grads, etas, pars, reused, data)
            etas["psi"] = -grads["psi"] * rho[0] ** 2 + beta * etas["psi"]
            etas["q"] = -grads["q"] * rho[1] ** 2 + beta * etas["q"]

        # step length
        alpha, top, bottom = calc_alpha(vars, grads, etas, pars, reused, data)

        # debug approxmation
        plot_debug(vars, etas, pars, top, bottom, alpha, data, i)
        error_debug(vars, pars, reused, data, alpha, i)
        vars["psi"] += alpha * etas["psi"]
        vars["q"] += alpha * etas["q"]
    return vars

# parameters
pars = {"niter": 128, "err_step": 1, "vis_step": 8}
pars["rho"] = [1, 20]

# variables
vars = {}
vars["psi"] = psi_init.copy()
vars["q"] = q_init.copy()
vars["table"] = pd.DataFrame(columns=["iter", "err", "alpha", "time"])

# reconstruction
vars = BH(data, vars, pars)

# results
erra = vars["table"]["err"].values
plt.title('error')
plt.xlabel('iter')
plt.plot(erra)
plt.yscale("log")
plt.grid()
plt.show()

alphaa = vars["table"]["alpha"].values
plt.title('alpha')
plt.xlabel('iter')
plt.plot(alphaa)
plt.grid()
plt.show()
